In [1]:
!pip install python-dotenv openai

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
text = """
Machine learning is a branch of artificial intelligence.

It allows computers to learn patterns from data.

Linear regression predicts continuous values.

Logistic regression predicts classes.

Decision trees split data into branches.

Random forests combine multiple trees.
"""

In [3]:
chunk_size = 100

chunks = []

for i in range(0, len(text), chunk_size):

    chunk = text[i:i+chunk_size]

    chunks.append(chunk)

In [4]:
for index, chunk in enumerate(chunks):

    print("----------------")

    print("Chunk:", index)

    print(chunk)

----------------
Chunk: 0

Machine learning is a branch of artificial intelligence.

It allows computers to learn patterns fro
----------------
Chunk: 1
m data.

Linear regression predicts continuous values.

Logistic regression predicts classes.

Decis
----------------
Chunk: 2
ion trees split data into branches.

Random forests combine multiple trees.



In [5]:
documents = [
    "Machine learning is amazing",
    "Python is a programming language",
    "Artificial intelligence uses data",
    "Linear regression predicts price"
]

In [6]:
from sklearn.feature_extraction.text import TfidfVectorizer
vectorizer = TfidfVectorizer()
vectors = vectorizer.fit_transform(documents)


In [7]:
print(vectorizer.get_feature_names_out())

['amazing' 'artificial' 'data' 'intelligence' 'is' 'language' 'learning'
 'linear' 'machine' 'predicts' 'price' 'programming' 'python' 'regression'
 'uses']


In [8]:
print(vectors.toarray())

[[0.52547275 0.         0.         0.         0.41428875 0.
  0.52547275 0.         0.52547275 0.         0.         0.
  0.         0.         0.        ]
 [0.         0.         0.         0.         0.41428875 0.52547275
  0.         0.         0.         0.         0.         0.52547275
  0.52547275 0.         0.        ]
 [0.         0.5        0.5        0.5        0.         0.
  0.         0.         0.         0.         0.         0.
  0.         0.         0.5       ]
 [0.         0.         0.         0.         0.         0.
  0.         0.5        0.         0.5        0.5        0.
  0.         0.5        0.        ]]


In [9]:
query = ["Linear regression predicts house prices"]

query_vector = vectorizer.transform(query)

In [10]:
from sklearn.metrics.pairwise import cosine_similarity

scores = cosine_similarity(query_vector, vectors)

In [11]:
print(scores)

[[0.        0.        0.        0.8660254]]


In [12]:
best_index = scores.argmax()

print(documents[best_index])

Linear regression predicts price


In [13]:
!pip install sentence-transformers faiss-cpu openai numpy

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
from openai import OpenAI
from dotenv import load_dotenv
import os

load_dotenv()

client = OpenAI(
    base_url=os.getenv("AZURE_AI_FOUNDRY_ENDPOINT"),
    api_key=os.getenv("AZURE_AI_FOUNDRY_KEY")
)

In [13]:
# 1. Naya sahi import
from google import genai
from dotenv import load_dotenv
import os

load_dotenv()

# 2. Pehle jahan genai.configure tha, wahan ab client banta hai
client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))

# 3. Pehle jahan genai.GenerativeModel tha, ab direct content generate hota hai
response = client.models.generate_content(
    model="gemini-3-flash-preview",
    contents="Say hello in one sentence."
)

print(response.text)

Hello, I hope you are having a wonderful day!


In [14]:
# requirements: sentence-transformers, faiss-cpu, openai
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np


# Step 1: Prepare documents
documents = [
    "Our refund policy: 30 days, full refund with receipt.",
    "Shipping takes 3-5 business days for domestic orders.",
    "We accept Visa, Mastercard, and PayPal.",
    "Customer support: support@example.com or call 1-800-HELP"
]


In [15]:
# Step 2: Create embeddings
model = SentenceTransformer('all-MiniLM-L6-v2')
embedding = model.encode(documents)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2595.81it/s]


In [16]:
# Step 3: Build FAISS index
dimension = embedding.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(np.array(embedding))

In [17]:
# Step 4: Retrieval function
def retrieve(query, k=2):
    query_embedding = model.encode([query])
    distances, indices = index.search(query_embedding, k)
    return [documents[i] for i in indices[0]]

In [20]:
# Step 5: RAG function
def rag_query(question):
    # Retrieve relevant docs
    context = retrieve(question)

    # Create prompt
    prompt = f"""Answer the question based only on this context:

Context:
{chr(10).join(context)}

Question: {question}

Answer:"""

    # Generate response
 

    response = client.models.generate_content(
        model="Phi-4-reasoning",
       # messages=[{"role": "user", "content": prompt}]
       messages=[
           
            {
                "role": "user", 
                "content": prompt
            }
        ]
    )

    return response.choices[0].message.content


In [21]:
# Test it
print(rag_query("How long does shipping take?"))

TypeError: Models.generate_content() got an unexpected keyword argument 'messages'